# 0) Grouped Dataset Preparation (Colab)

Duplicate-aware, taxonomy-aware dataset preparation before Notebook 2 training.


In [ ]:
from google.colab import drive, userdata

import os
import subprocess
import sys

drive.mount('/content/drive')

HF_TOKEN_NAMES = ("HF_TOKEN", "HUGGINGFACE_TOKEN", "HUGGINGFACE_HUB_TOKEN")
GITHUB_TOKEN_NAMES = ("GH_TOKEN", "GITHUB_TOKEN")

def _resolve_token(token_names: tuple[str, ...], canonical_env_name: str) -> str | None:
    for env_name in token_names:
        token = str(os.environ.get(env_name, "")).strip()
        if token:
            os.environ.setdefault(canonical_env_name, token)
            return token
    for secret_name in token_names:
        try:
            token = str(userdata.get(secret_name) or "").strip()
        except Exception:
            token = ""
        if token:
            os.environ[canonical_env_name] = token
            return token
    return None

hf_token = _resolve_token(HF_TOKEN_NAMES, "HF_TOKEN")
gh_token = _resolve_token(GITHUB_TOKEN_NAMES, "GH_TOKEN")
if gh_token:
    print("[GIT] GitHub token loaded from env/Colab secrets.")
else:
    print("[GIT] No GitHub token found. Auto-push will be skipped unless GH_TOKEN or GITHUB_TOKEN is set.")

if not hf_token:
    print("[HF] No token found. Set a Colab secret or env var named HF_TOKEN before training.")
else:
    try:
        from huggingface_hub import HfApi, login
    except Exception:
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", "huggingface_hub"], check=False)
        from huggingface_hub import HfApi, login

    try:
        login(token=hf_token, add_to_git_credential=False)
        profile = dict(HfApi(token=hf_token).whoami() or {})
        username = str(
            profile.get("name")
            or profile.get("fullname")
            or profile.get("email")
            or profile.get("user")
            or "authenticated user"
        )
        print(f"[HF] Authenticated as {username}")
    except Exception as exc:
        print(f"[HF] Authentication check failed: {exc}")
        print("[HF] Training may fail when gated models need authentication.")

In [ ]:
import io
import json
import os
import random
import shutil
import sys
import subprocess
import time
from contextlib import contextmanager
from pathlib import Path
from datetime import datetime, timezone
from urllib.parse import urlsplit, urlunsplit

import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import torch

# --- Bootstrap: resolve repo root, install deps ---
GITHUB_TOKEN_NAMES = ('GH_TOKEN', 'GITHUB_TOKEN')
CLONE_TARGET = Path('/content/bitirmeprojesi')
REPO_URL = os.environ.get('AADS_REPO_URL', 'https://github.com/EfeErim/bitirmeprojesi.git')
COMMON_REPO_CANDIDATES = (
    Path('/content/bitirme projesi'),
    CLONE_TARGET,
    Path('/content/aads_ulora'),
    Path('/content/drive/MyDrive/bitirme projesi'),
    Path('/content/drive/MyDrive/bitirmeprojesi'),
)

def _is_repo_root(path: Path) -> bool:
    return (path / 'src').is_dir() and (path / 'config').is_dir() and (path / 'scripts').is_dir()

def _running_in_colab_inline() -> bool:
    try:
        import google.colab  # noqa: F401
    except Exception:
        return False
    return True

def _mount_drive_inline() -> None:
    if not _running_in_colab_inline():
        return
    try:
        from google.colab import drive
        drive.mount('/content/drive', force_remount=False)
    except Exception as exc:
        print(f'Drive mount skipped during bootstrap: {exc}')

def _resolve_colab_secret_inline(secret_name: str) -> str:
    if not _running_in_colab_inline():
        return ''
    try:
        from google.colab import userdata
        return str(userdata.get(secret_name) or '').strip()
    except Exception:
        return ''

def _resolve_github_token_inline() -> str:
    for env_name in GITHUB_TOKEN_NAMES:
        token = str(os.environ.get(env_name, '')).strip()
        if token:
            os.environ.setdefault('GH_TOKEN', token)
            return token
    for secret_name in GITHUB_TOKEN_NAMES:
        token = _resolve_colab_secret_inline(secret_name)
        if token:
            os.environ['GH_TOKEN'] = token
            return token
    return ''

def _build_repo_access_url(repo_url: str, token: str) -> str:
    if not token:
        return repo_url
    parsed = urlsplit(str(repo_url or '').strip())
    if parsed.scheme != 'https' or not parsed.netloc:
        return repo_url
    netloc = parsed.netloc.split('@', 1)[-1]
    return urlunsplit((parsed.scheme, f'{token}@{netloc}', parsed.path, parsed.query, parsed.fragment))

def _find_repo_root_inline() -> Path | None:
    for raw in (os.environ.get('AADS_REPO_ROOT'), os.environ.get('REPO_ROOT')):
        if not raw:
            continue
        candidate = Path(raw).expanduser().resolve()
        if _is_repo_root(candidate):
            return candidate
    cwd = Path.cwd().resolve()
    for candidate in [cwd, *cwd.parents]:
        if _is_repo_root(candidate):
            return candidate
    for candidate in COMMON_REPO_CANDIDATES:
        if _is_repo_root(candidate):
            return candidate
    if CLONE_TARGET.exists() and any(CLONE_TARGET.iterdir()):
        for child in CLONE_TARGET.iterdir():
            if child.is_dir() and _is_repo_root(child):
                return child
    return None

def _ensure_repo_root_for_bootstrap() -> Path:
    repo_root = _find_repo_root_inline()
    if repo_root is not None:
        return repo_root
    _mount_drive_inline()
    repo_root = _find_repo_root_inline()
    if repo_root is not None:
        return repo_root
    clone_url = _build_repo_access_url(REPO_URL, _resolve_github_token_inline())
    CLONE_TARGET.parent.mkdir(parents=True, exist_ok=True)
    completed = subprocess.run(
        ['git', 'clone', '--depth', '1', clone_url, str(CLONE_TARGET)],
        check=False,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True,
    )
    if completed.stdout:
        print(completed.stdout)
    repo_root = _find_repo_root_inline()
    if repo_root is not None:
        return repo_root
    raise RuntimeError(
        'Repository bootstrap failed. Set AADS_REPO_ROOT to an existing checkout, or provide '
        'GH_TOKEN/GITHUB_TOKEN for private GitHub auto-clone.'
    )

BOOTSTRAP_ROOT = _ensure_repo_root_for_bootstrap()
os.chdir(BOOTSTRAP_ROOT)
if str(BOOTSTRAP_ROOT) not in sys.path:
    sys.path.insert(0, str(BOOTSTRAP_ROOT))

from scripts.colab_repo_bootstrap import (
    export_current_colab_notebook,
    install_colab_requirements,
    login_and_check_hf_token,
    mirror_checkpoint_state_to_repo,
    mirror_path_to_repo,
    mount_drive_if_available,
    push_repo_run_to_github,
    resolve_github_token,
    resolve_hf_token,
    resolve_repo_root,
    running_in_colab,
)

IN_COLAB = running_in_colab()
if IN_COLAB:
    mount_drive_if_available(force_remount=False)

ROOT = resolve_repo_root()
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

install_colab_requirements(ROOT / 'colab_notebooks' / 'requirements_colab.txt', IN_COLAB)

from scripts.colab_checkpointing import TrainingCheckpointManager
from scripts.colab_live_telemetry import ColabLiveTelemetry
from src.core.config_manager import ConfigurationManager


def _install_capture_cell_output_compat() -> bool:
    if hasattr(ColabLiveTelemetry, 'capture_cell_output'):
        return False

    def _slugify_capture_name(value: str) -> str:
        slug = ''.join(ch.lower() if ch.isalnum() else '_' for ch in str(value or '').strip())
        while '__' in slug:
            slug = slug.replace('__', '_')
        slug = slug.strip('_')
        return slug or 'cell'

    class _CompatTeeTextStream:
        def __init__(self, *streams):
            self._streams = [stream for stream in streams if stream is not None]

        def write(self, data):
            text = str(data)
            for stream in self._streams:
                stream.write(text)
            return len(text)

        def flush(self):
            for stream in self._streams:
                flush = getattr(stream, 'flush', None)
                if callable(flush):
                    flush()

        def isatty(self):
            return any(bool(getattr(stream, 'isatty', lambda: False)()) for stream in self._streams)

        def writable(self):
            return True

CAPTURE_CELL_OUTPUT_COMPAT_INSTALLED = _install_capture_cell_output_compat()
RUN_ID = datetime.now().strftime('%Y%m%d_%H%M%S_%f')
NOTEBOOK_FILENAME = '0_grouped_dataset_preparation.executed.ipynb'
REPO_RUN_DIR = ROOT / 'runs' / RUN_ID
REPO_NOTEBOOK_OUTPUT_PATH = REPO_RUN_DIR / 'notebooks' / NOTEBOOK_FILENAME
LOCAL_OUTPUT_DIR = ROOT / 'outputs' / 'colab_notebook_data_prep'
REPO_OUTPUT_DIR = REPO_RUN_DIR / 'outputs' / 'colab_notebook_data_prep'
REPO_TELEMETRY_DIR = REPO_RUN_DIR / 'telemetry'
REPO_CHECKPOINT_STATE_DIR = REPO_RUN_DIR / 'checkpoint_state'

TELEMETRY = ColabLiveTelemetry(
    notebook_name='0_grouped_dataset_preparation.ipynb',
    run_id=RUN_ID,
)
CHECKPOINT_ROOT = TELEMETRY.drive_run_dir if TELEMETRY.drive_run_dir.exists() else TELEMETRY.local_run_dir
CHECKPOINT_MANAGER = TrainingCheckpointManager(CHECKPOINT_ROOT, retention=3)
DEVICE = str(ConfigurationManager(config_dir=str(ROOT / 'config'), environment='colab').load_all_configs().get('training', {}).get('continual', {}).get('device', 'cuda'))

LOCAL_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
REPO_NOTEBOOK_OUTPUT_PATH.parent.mkdir(parents=True, exist_ok=True)

def rt(message: str, *, phase: str = 'notebook') -> None:
    text = str(message)
    print(text)
    TELEMETRY.emit_log(text, phase=phase, level='info')

def save_run_outputs_to_repo() -> dict[str, str]:
    exports: dict[str, str] = {}

    mirrored_outputs = mirror_path_to_repo(LOCAL_OUTPUT_DIR, REPO_OUTPUT_DIR)
    if mirrored_outputs is not None:
        exports['outputs'] = str(mirrored_outputs)

    telemetry_source = TELEMETRY.drive_run_dir if TELEMETRY.drive_run_dir.exists() else TELEMETRY.local_run_dir
    mirrored_telemetry = mirror_path_to_repo(telemetry_source, REPO_TELEMETRY_DIR)
    if mirrored_telemetry is not None:
        exports['telemetry'] = str(mirrored_telemetry)

    mirrored_checkpoint_state = mirror_checkpoint_state_to_repo(CHECKPOINT_ROOT, REPO_CHECKPOINT_STATE_DIR)
    if mirrored_checkpoint_state is not None:
        exports['checkpoint_state'] = str(mirrored_checkpoint_state)

    return exports

def _copy_path_to_drive_exports(source_path: Path, relative_name: str) -> Path | None:
    source = Path(source_path).expanduser()
    if not source.exists():
        return None
    drive_root = TELEMETRY.drive_run_dir if TELEMETRY.drive_run_dir.exists() else None
    if drive_root is None:
        return None
    target = drive_root / 'artifacts' / relative_name
    if source.is_dir():
        if target.exists():
            shutil.rmtree(target)
        shutil.copytree(source, target)
    else:
        target.parent.mkdir(parents=True, exist_ok=True)
        shutil.copy2(source, target)
    return target

TELEMETRY.configure_repo_output_export(
    output_dir=REPO_OUTPUT_DIR,
    notebook_filename=NOTEBOOK_FILENAME,
    export_notebook_fn=export_current_colab_notebook,
)
TELEMETRY.update_latest({'phase': 'bootstrap_ready', 'run_id': RUN_ID})
print(f'[BOOTSTRAP] run_id={RUN_ID} capture_cell_output_compat={CAPTURE_CELL_OUTPUT_COMPAT_INSTALLED}')


In [ ]:
with TELEMETRY.capture_cell_output("Cell 3: Parameters"):
    # --- Data prep parameters ---
    # Edit these directly, then run the remaining cells top-to-bottom.

    # DATASET_ROOT: flat class-root dataset containing one folder per supported class.
    DATASET_ROOT = "data/class_root_dataset"

    # CROP_NAME: crop key used for taxonomy-aware class normalization.
    CROP_NAME = "tomato"

    # PREP_ARTIFACT_ROOT: where audit manifests and review reports are written.
    # Keep this local for speed when possible; Notebook 0 mirrors the artifacts to Drive telemetry automatically.
    PREP_ARTIFACT_ROOT = "outputs/colab_notebook_data_prep/artifacts"

    # PREPARED_RUNTIME_ROOT: where the grouped runtime dataset is materialized when approved.
    # A local workspace path is faster than writing directly into Drive; the prepared runtime dataset is mirrored to Drive too.
    PREPARED_RUNTIME_ROOT = "data/prepared_runtime_datasets"

    # MATERIALIZE_AFTER_REVIEW: leave False for audit-only mode; switch to True after review passes.
    MATERIALIZE_AFTER_REVIEW = False

    # DEVICE: embedding device used for DINOv3 and BioCLIP similarity features.
    DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

    # EMBEDDING_BATCH_SIZE: image batch size for embedding extraction.
    # 32 is a safe GPU default; move to 48 only if A100/H100 memory headroom is clearly available.
    EMBEDDING_BATCH_SIZE = 32 if DEVICE == "cuda" else 8

    # NEIGHBORS: nearest neighbors per image used for same-class candidate mining.
    # Lower values reduce runtime; 4 is the faster default for large audits.
    NEIGHBORS = 4

    # PREP_DINOV3_MODEL_ID: override with a smaller DINOv3 checkpoint for faster exploratory audits if needed.
    PREP_DINOV3_MODEL_ID = "facebook/dinov3-vitl16-pretrain-lvd1689m"
    PREP_BIOCLIP_MODEL_ID = "imageomics/bioclip-2.5-vith14"

    run_id = RUN_ID
    STATE = {
        "validated": False,
        "audit_summary": None,
        "artifact_root": None,
        "runtime_dataset_root": None,
        "dataset_root": None,
    }
    print(f"[PARAMS] run_id={run_id} crop={CROP_NAME} dataset_root={DATASET_ROOT}")


In [ ]:
with TELEMETRY.capture_cell_output("Cell 4: Dataset Audit"):
    from scripts.prepare_grouped_runtime_dataset import build_grouped_dataset_plan

    dataset_root = Path(DATASET_ROOT).expanduser()
    if not dataset_root.is_absolute():
        dataset_root = (ROOT / dataset_root).resolve()
    if not dataset_root.is_dir():
        raise RuntimeError(f"Dataset root not found: {dataset_root}")

    artifact_root = Path(PREP_ARTIFACT_ROOT).expanduser()
    if not artifact_root.is_absolute():
        artifact_root = (ROOT / artifact_root).resolve()

    summary = build_grouped_dataset_plan(
        class_root=dataset_root,
        crop_name=CROP_NAME,
        artifact_root=artifact_root,
        taxonomy_path=ROOT / "config" / "plant_taxonomy.json",
        dino_model_id=PREP_DINOV3_MODEL_ID,
        bioclip_model_id=PREP_BIOCLIP_MODEL_ID,
        device=DEVICE,
        batch_size=EMBEDDING_BATCH_SIZE,
        neighbors=NEIGHBORS,
        progress_fn=lambda message: print(f"[PREP] {message}"),
    )

    STATE["validated"] = True
    STATE["audit_summary"] = summary
    STATE["artifact_root"] = artifact_root
    STATE["dataset_root"] = dataset_root
    print(json.dumps(summary.get("summary", {}), indent=2))
    print(f"[PREP] runtime_ready={summary.get('runtime_ready')} artifact_root={artifact_root}")
    if summary.get("blocking_issues"):
        print("[PREP] blocking issues:")
        for item in summary["blocking_issues"]:
            print(f"  - {item}")
    drive_artifact_root = _copy_path_to_drive_exports(artifact_root, 'data_prep_artifacts')
    if drive_artifact_root is not None:
        print(f"[PREP] Drive artifact mirror updated at {drive_artifact_root}")

    TELEMETRY.update_latest(
        {
            "phase": "data_prep_audited",
            "dataset_root": str(dataset_root),
            "artifact_root": str(artifact_root),
            "drive_artifact_root": ("" if drive_artifact_root is None else str(drive_artifact_root)),
            "runtime_ready": bool(summary.get("runtime_ready")),
        }
    )


In [ ]:
with TELEMETRY.capture_cell_output("Cell 5: Materialize Runtime Dataset"):
    from scripts.prepare_grouped_runtime_dataset import materialize_grouped_runtime_dataset

    if not STATE.get("validated") or STATE.get("audit_summary") is None:
        raise RuntimeError("Run the dataset audit cell first.")

    summary = STATE["audit_summary"]
    if not MATERIALIZE_AFTER_REVIEW:
        print("[PREP] MATERIALIZE_AFTER_REVIEW is False. Audit artifacts are ready for review.")
    else:
        if not summary.get("runtime_ready"):
            raise RuntimeError("Audit has blocking issues. Resolve them before materialization.")
        runtime_root = Path(PREPARED_RUNTIME_ROOT).expanduser()
        if not runtime_root.is_absolute():
            runtime_root = (ROOT / runtime_root).resolve()
        runtime_dataset_root = materialize_grouped_runtime_dataset(
            class_root=STATE["dataset_root"],
            crop_name=CROP_NAME,
            artifact_root=STATE["artifact_root"],
            runtime_root=runtime_root,
            materialization_strategy="auto",
        )
        STATE["runtime_dataset_root"] = runtime_dataset_root
        print(f"[PREP] Prepared runtime dataset written under {runtime_dataset_root / CROP_NAME}")
        drive_runtime_root = _copy_path_to_drive_exports(runtime_dataset_root / CROP_NAME, f'prepared_runtime_datasets/{CROP_NAME}')
        if drive_runtime_root is not None:
            print(f"[PREP] Drive runtime mirror updated at {drive_runtime_root}")
        TELEMETRY.update_latest(
            {
                "phase": "data_prep_materialized",
                "runtime_dataset_root": str(runtime_dataset_root),
                "drive_runtime_root": ("" if drive_runtime_root is None else str(drive_runtime_root)),
            }
        )
